In [6]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OrdinalEncoder
import optuna

# 1_2. Survival ML: XGBoost

https://xgboost.readthedocs.io/en/stable/tutorials/aft_survival_analysis.html

## Deciduous

In [7]:
deciduous_teeth = pd.read_csv("../data/deciduous_teeth.csv", sep=";")

X = covars encoded

y = lower & upper bound

### Preprocess

In [8]:
deciduous_teeth.loc[((deciduous_teeth["LOWER"].isna()), "LOWER")] = 0
deciduous_teeth.loc[((deciduous_teeth["UPPER"].isna()), "UPPER")] = np.inf

max/mand, side, sex --> 0/1

sens, breed, litter --> drop

breed size --> ordinal

skull type, breed clade --> ohe

In [9]:
oe = OrdinalEncoder(categories=[["Small", "Medium", "Large", "Giant"]])
deciduous_teeth["BREED_SIZE"] = oe.fit_transform(deciduous_teeth[["BREED_SIZE"]])

oe = OrdinalEncoder(categories=[["MAND", "MAX"]])
deciduous_teeth["MAX_MAND"] = oe.fit_transform(deciduous_teeth[["MAX_MAND"]])

oe = OrdinalEncoder(categories=[["L", "R"]])
deciduous_teeth["SIDE"] = oe.fit_transform(deciduous_teeth[["SIDE"]])

oe = OrdinalEncoder(
    categories=[["M", "F"]], unknown_value=np.nan, handle_unknown="use_encoded_value"
)
deciduous_teeth["SEX"] = oe.fit_transform(deciduous_teeth[["SEX"]])

In [10]:
one_hot = pd.get_dummies(deciduous_teeth[["SKULL_TYPE", "BREED_CLADE"]])
deciduous_teeth.drop(
    columns=["SKULL_TYPE", "BREED_CLADE", "BREED"],
    inplace=True,
)
deciduous_teeth = deciduous_teeth.join(one_hot)

In [16]:
deciduous_teeth.columns

Index(['DOG', 'TOOTH_NUMBER', 'MAX_MAND', 'SIDE', 'POSITION', 'DEVELOPM_STAGE',
       'LOWER', 'UPPER', 'CENS', 'SEX', 'BREED_SIZE', 'SKULL_TYPE_Brachy',
       'SKULL_TYPE_Doligo', 'SKULL_TYPE_Meso', 'SKULL_TYPE_Meso*',
       'SKULL_TYPE_Meso**', 'BREED_CLADE_A', 'BREED_CLADE_B', 'BREED_CLADE_D',
       'BREED_CLADE_E', 'BREED_CLADE_F', 'BREED_CLADE_H', 'BREED_CLADE_I',
       'BREED_CLADE_L', 'BREED_CLADE_M', 'BREED_CLADE_O', 'BREED_CLADE_P',
       'BREED_CLADE_Q', 'BREED_CLADE_R', 'BREED_CLADE_S', 'BREED_CLADE_T',
       'BREED_CLADE_V', 'BREED_CLADE_W', 'BREED_CLADE_X', 'BREED_CLADE_Y'],
      dtype='object')

### Define hyperparams

hyperparam search met CV

In [17]:
def make_dmatrix(df, features):
    X = df[features].values
    y_lower = np.array(df["LOWER"])
    y_upper = np.array(df["UPPER"])

    dmatrix = xgb.DMatrix(X, feature_names=features)

    dmatrix.set_float_info("label_lower_bound", y_lower)
    dmatrix.set_float_info("label_upper_bound", y_upper)

    return dmatrix

In [19]:
# Define hyperparameter search space
base_params = {
    "verbosity": 0,
    "objective": "survival:aft",
    "eval_metric": "aft-nloglik",
    "tree_method": "hist",
    "monotone_constraints": {"DEVELOPM_STAGE": 1},
}  # Hyperparameters common to all trials

feature_combos = [
    list(
        deciduous_teeth.columns.difference(
            ["LOWER", "UPPER", "DOG", "CENS"], sort=False
        )
    ),  # alle features
    [
        "DEVELOPM_STAGE",
        "MAX_MAND",
        "SIDE",
        "POSITION",
        "SEX",
        "BREED_SIZE",
        "SKULL_TYPE_Brachy",
        "SKULL_TYPE_Doligo",
        "SKULL_TYPE_Meso",
        "SKULL_TYPE_Meso*",
        "SKULL_TYPE_Meso**",
        "BREED_CLADE_A",
        "BREED_CLADE_B",
        "BREED_CLADE_D",
        "BREED_CLADE_E",
        "BREED_CLADE_F",
        "BREED_CLADE_H",
        "BREED_CLADE_I",
        "BREED_CLADE_L",
        "BREED_CLADE_M",
        "BREED_CLADE_O",
        "BREED_CLADE_P",
        "BREED_CLADE_Q",
        "BREED_CLADE_R",
        "BREED_CLADE_S",
        "BREED_CLADE_T",
        "BREED_CLADE_V",
        "BREED_CLADE_W",
        "BREED_CLADE_X",
        "BREED_CLADE_Y",
    ],
    [
        "DEVELOPM_STAGE",
        "TOOTH_NUMBER",
        "SEX",
        "BREED_SIZE",
        "SKULL_TYPE_Brachy",
        "SKULL_TYPE_Doligo",
        "SKULL_TYPE_Meso",
        "SKULL_TYPE_Meso*",
        "SKULL_TYPE_Meso**",
        "BREED_CLADE_A",
        "BREED_CLADE_B",
        "BREED_CLADE_D",
        "BREED_CLADE_E",
        "BREED_CLADE_F",
        "BREED_CLADE_H",
        "BREED_CLADE_I",
        "BREED_CLADE_L",
        "BREED_CLADE_M",
        "BREED_CLADE_O",
        "BREED_CLADE_P",
        "BREED_CLADE_Q",
        "BREED_CLADE_R",
        "BREED_CLADE_S",
        "BREED_CLADE_T",
        "BREED_CLADE_V",
        "BREED_CLADE_W",
        "BREED_CLADE_X",
        "BREED_CLADE_Y",
    ],
    [
        "TOOTH_NUMBER",
        "MAX_MAND",
        "SIDE",
        "POSITION",
        "DEVELOPM_STAGE",
        "BREED_SIZE",
        "SKULL_TYPE_Brachy",
        "SKULL_TYPE_Doligo",
        "SKULL_TYPE_Meso",
        "SKULL_TYPE_Meso*",
        "SKULL_TYPE_Meso**",
    ],
    [
        "DEVELOPM_STAGE",
        "TOOTH_NUMBER",
        "BREED_SIZE",
        "SKULL_TYPE_Brachy",
        "SKULL_TYPE_Doligo",
        "SKULL_TYPE_Meso",
        "SKULL_TYPE_Meso*",
        "SKULL_TYPE_Meso**",
    ],
]

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3
test_preds = {}

lodo = StratifiedGroupKFold(N_OUTER_FOLDS, shuffle=True, random_state=42)
for i, (train_index, test_index) in enumerate(
    lodo.split(
        X=deciduous_teeth, y=deciduous_teeth["CENS"], groups=deciduous_teeth["DOG"]
    )
):
    print(i)
    train_outer = deciduous_teeth.iloc[train_index]
    test_outer = deciduous_teeth.iloc[test_index]

    def objective(trial):
        feature_selection = trial.suggest_int(
            "feature_selection", 0, len(feature_combos) - 1
        )

        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
            "aft_loss_distribution": trial.suggest_categorical(
                "aft_loss_distribution", ["normal", "logistic", "extreme"]
            ),
            "aft_loss_distribution_scale": trial.suggest_float(
                "aft_loss_distribution_scale", 0.01, 5, log=True
            ),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "min_child_weight": trial.suggest_float(
                "min_child_weight", 1, 10, log=True
            ),
            "lambda": trial.suggest_float("lambda", 1e-3, 10.0),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        }  # Search space
        params.update(base_params)
        pruning_callback = optuna.integration.XGBoostPruningCallback(
            trial, "valid-aft-nloglik"
        )

        # CV
        train_events = deciduous_teeth.loc[train_index, "CENS"]
        train_ids = deciduous_teeth.loc[train_index, "DOG"]
        best_iters = []
        sgkf = StratifiedGroupKFold(
            n_splits=N_INNER_FOLDS, shuffle=True, random_state=42
        )
        CV_scores = []

        for idx, (train_idx_inner, test_idx_inner) in enumerate(
            sgkf.split(train_outer, train_events, train_ids)
        ):
            train_inner, test_inner = (
                train_outer.iloc[train_idx_inner],
                train_outer.iloc[test_idx_inner],
            )

            dtrain_inner = make_dmatrix(train_inner, feature_combos[feature_selection])
            dtest_inner = make_dmatrix(test_inner, feature_combos[feature_selection])

            if idx == 0:
                bst_inner = xgb.train(
                    params,
                    dtrain_inner,
                    num_boost_round=1000,
                    evals=[(dtrain_inner, "train"), (dtest_inner, "valid")],
                    early_stopping_rounds=100,
                    verbose_eval=False,
                    callbacks=[pruning_callback],
                )
            else:
                bst_inner = xgb.train(
                    params,
                    dtrain_inner,
                    num_boost_round=1000,
                    evals=[(dtrain_inner, "train"), (dtest_inner, "valid")],
                    early_stopping_rounds=100,
                    verbose_eval=False,
                )

            best_iters.append(bst_inner.best_iteration)
            CV_scores.append(bst_inner.best_score)

        trial.set_user_attr("best_iteration", int(np.median(best_iters)))
        return np.mean(CV_scores)

    # Optuna for hyperparam tuning
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=100, show_progress_bar=False)

    # get best params out of inner loop
    best_iter = study.best_trial.user_attrs["best_iteration"]
    best_params = {}
    best_params.update(base_params)
    best_params.update(study.best_trial.params)
    best_features = best_params.pop("feature_selection")
    best_dist = best_params["aft_loss_distribution"]
    best_scale = best_params["aft_loss_distribution_scale"]

    # dmatrix maken
    dtrain_outer = make_dmatrix(train_outer, feature_combos[best_features])
    dtest_outer = make_dmatrix(test_outer, feature_combos[best_features])

    # outer model trainen
    bst_outer = xgb.train(
        best_params,
        dtrain_outer,
        num_boost_round=best_iter,
        verbose_eval=False,
    )
    preds_outer = bst_outer.predict(dtest_outer, output_margin=True)
    # Store by original indices
    for idx, pred_outer in zip(test_index, preds_outer):
        test_preds[idx] = {
            "pred": pred_outer,
            "dist": best_dist,
            "scale": best_scale,
        }

[I 2026-03-11 08:59:56,720] A new study created in memory with name: no-name-3e64fa48-1df6-45ad-a548-04cfbee21cbb


0


[I 2026-03-11 09:00:23,861] Trial 0 finished with value: 1.141228884439217 and parameters: {'feature_selection': 2, 'learning_rate': 0.007564475862334781, 'aft_loss_distribution': 'logistic', 'aft_loss_distribution_scale': 0.7692210939423935, 'max_depth': 6, 'min_child_weight': 7.499723911895367, 'lambda': 8.207776020974878, 'subsample': 0.9963629547917874, 'colsample_bylevel': 0.6753784679566348}. Best is trial 0 with value: 1.141228884439217.
[I 2026-03-11 09:00:44,639] Trial 1 finished with value: 1.1303923486697462 and parameters: {'feature_selection': 3, 'learning_rate': 0.007717446182194615, 'aft_loss_distribution': 'logistic', 'aft_loss_distribution_scale': 0.6304096855488238, 'max_depth': 5, 'min_child_weight': 3.8135416140830847, 'lambda': 3.6227009241426296, 'subsample': 0.7515379570457237, 'colsample_bylevel': 0.7278053486189885}. Best is trial 1 with value: 1.1303923486697462.
[I 2026-03-11 09:00:47,513] Trial 2 finished with value: 11.763206882379675 and parameters: {'feat

1


[I 2026-03-11 09:02:21,951] Trial 0 finished with value: 1.5461738905941633 and parameters: {'feature_selection': 0, 'learning_rate': 0.015710318507124586, 'aft_loss_distribution': 'logistic', 'aft_loss_distribution_scale': 2.6876152960595654, 'max_depth': 6, 'min_child_weight': 5.444845902161454, 'lambda': 3.9245968783513163, 'subsample': 0.6621064398526915, 'colsample_bylevel': 0.7563071071377288}. Best is trial 0 with value: 1.5461738905941633.
[I 2026-03-11 09:02:31,045] Trial 1 finished with value: 0.842002662695155 and parameters: {'feature_selection': 0, 'learning_rate': 0.06580986356411453, 'aft_loss_distribution': 'normal', 'aft_loss_distribution_scale': 0.4607718079040645, 'max_depth': 3, 'min_child_weight': 6.386631333913573, 'lambda': 0.6813303696803624, 'subsample': 0.7741648858733746, 'colsample_bylevel': 0.5375782006932497}. Best is trial 1 with value: 0.842002662695155.
[I 2026-03-11 09:02:44,920] Trial 2 finished with value: 1.0540088890122856 and parameters: {'feature

...

[I 2026-03-11 09:04:09,583] Trial 0 finished with value: 1.5136365074096638 and parameters: {'feature_selection': 2, 'learning_rate': 0.008200011223166126, 'aft_loss_distribution': 'extreme', 'aft_loss_distribution_scale': 4.028324496593137, 'max_depth': 4, 'min_child_weight': 3.7079425889566204, 'lambda': 1.5258642086027636, 'subsample': 0.9923446319838587, 'colsample_bylevel': 0.7069090544391057}. Best is trial 0 with value: 1.5136365074096638.
[I 2026-03-11 09:04:21,030] Trial 1 finished with value: 1.5907974974137737 and parameters: {'feature_selection': 4, 'learning_rate': 0.11580641788904147, 'aft_loss_distribution': 'normal', 'aft_loss_distribution_scale': 4.483121756499085, 'max_depth': 3, 'min_child_weight': 5.831899617849215, 'lambda': 3.8316336159980104, 'subsample': 0.9469622668152475, 'colsample_bylevel': 0.5880097079449987}. Best is trial 0 with value: 1.5136365074096638.
[I 2026-03-11 09:04:35,652] Trial 2 finished with value: 1.142809894877035 and parameters: {'feature_

3


[I 2026-03-11 09:07:10,892] Trial 0 finished with value: 11.539623002986405 and parameters: {'feature_selection': 3, 'learning_rate': 0.01054089760013215, 'aft_loss_distribution': 'extreme', 'aft_loss_distribution_scale': 0.18203384985725943, 'max_depth': 6, 'min_child_weight': 2.9009693829675, 'lambda': 6.709282163318904, 'subsample': 0.6622127383526426, 'colsample_bylevel': 0.8375967333584537}. Best is trial 0 with value: 11.539623002986405.
[I 2026-03-11 09:07:16,557] Trial 1 finished with value: 1.1659326265035472 and parameters: {'feature_selection': 1, 'learning_rate': 0.07001666218167803, 'aft_loss_distribution': 'normal', 'aft_loss_distribution_scale': 1.4452326206178152, 'max_depth': 8, 'min_child_weight': 1.1669014479247206, 'lambda': 6.71933131324955, 'subsample': 0.5552848864327372, 'colsample_bylevel': 0.8035792978428774}. Best is trial 1 with value: 1.1659326265035472.
[I 2026-03-11 09:07:32,842] Trial 2 finished with value: 1.1739019824921944 and parameters: {'feature_se

4


[I 2026-03-11 09:10:47,025] Trial 0 finished with value: 2.5425878344364112 and parameters: {'feature_selection': 3, 'learning_rate': 0.012153674740124755, 'aft_loss_distribution': 'logistic', 'aft_loss_distribution_scale': 0.09813272407576587, 'max_depth': 7, 'min_child_weight': 2.9258836380951783, 'lambda': 6.858221753623506, 'subsample': 0.8316199109974434, 'colsample_bylevel': 0.5909911559778259}. Best is trial 0 with value: 2.5425878344364112.
[I 2026-03-11 09:10:49,478] Trial 1 finished with value: 11.556509520706243 and parameters: {'feature_selection': 1, 'learning_rate': 0.0025590428819907375, 'aft_loss_distribution': 'normal', 'aft_loss_distribution_scale': 0.019007847819915703, 'max_depth': 3, 'min_child_weight': 9.06315309937716, 'lambda': 1.7821606614463847, 'subsample': 0.6680224092866273, 'colsample_bylevel': 0.7036967243747401}. Best is trial 0 with value: 2.5425878344364112.
[I 2026-03-11 09:11:10,261] Trial 2 finished with value: 1.5722058600394437 and parameters: {'f

In [15]:
# Convert to DataFrame in original order
test_preds_df = pd.DataFrame.from_dict(test_preds, orient="index").sort_index()

In [ ]:
test_preds_df.to_csv(r"preds/xgb_tooth_preds_dec.csv", sep=";")

## Permanent

In [20]:
permanent_teeth = pd.read_csv("../data/permanent_teeth.csv", sep=";")

### Preprocess

In [21]:
permanent_teeth.loc[((permanent_teeth["LOWER"].isna()), "LOWER")] = 0
permanent_teeth.loc[((permanent_teeth["UPPER"].isna()), "UPPER")] = np.inf

max/mand, side, sex --> 0/1

sens, breed, litter --> drop

breed size --> ordinal

skull type, breed clade --> ohe

In [22]:
oe = OrdinalEncoder(
    categories=[["Toy", "Small", "Medium", "Large", "Giant"]],
    unknown_value=np.nan,
    handle_unknown="use_encoded_value",
)
permanent_teeth["BREED_SIZE"] = oe.fit_transform(permanent_teeth[["BREED_SIZE"]])

oe = OrdinalEncoder(categories=[["MAND", "MAX"]])
permanent_teeth["MAX_MAND"] = oe.fit_transform(permanent_teeth[["MAX_MAND"]])

oe = OrdinalEncoder(categories=[["L", "R"]])
permanent_teeth["SIDE"] = oe.fit_transform(permanent_teeth[["SIDE"]])

oe = OrdinalEncoder(
    categories=[["M", "F"]], unknown_value=np.nan, handle_unknown="use_encoded_value"
)
permanent_teeth["SEX"] = oe.fit_transform(permanent_teeth[["SEX"]])

In [23]:
one_hot = pd.get_dummies(permanent_teeth[["SKULL_TYPE", "BREED_CLADE"]])
permanent_teeth.drop(
    columns=[
        "SKULL_TYPE",
        "BREED_CLADE",
        "BREED",
    ],
    inplace=True,
)
permanent_teeth = permanent_teeth.join(one_hot)

In [24]:
permanent_teeth.columns

Index(['DOG', 'TOOTH_NUMBER', 'MAX_MAND', 'SIDE', 'POSITION', 'DEVELOPM_STAGE',
       'LOWER', 'UPPER', 'CENS', 'SEX', 'BREED_SIZE', 'SKULL_TYPE_Brachy',
       'SKULL_TYPE_Doligo', 'SKULL_TYPE_Meso', 'SKULL_TYPE_Meso*',
       'SKULL_TYPE_Meso**', 'BREED_CLADE_A', 'BREED_CLADE_B', 'BREED_CLADE_D',
       'BREED_CLADE_E', 'BREED_CLADE_F', 'BREED_CLADE_H', 'BREED_CLADE_I',
       'BREED_CLADE_L', 'BREED_CLADE_M', 'BREED_CLADE_N', 'BREED_CLADE_O',
       'BREED_CLADE_P', 'BREED_CLADE_Q', 'BREED_CLADE_R', 'BREED_CLADE_S',
       'BREED_CLADE_T', 'BREED_CLADE_U', 'BREED_CLADE_V', 'BREED_CLADE_W',
       'BREED_CLADE_X', 'BREED_CLADE_Y'],
      dtype='object')

### Define hyperparams

hyperparam search met CV

In [25]:
# Define hyperparameter search space
base_params = {
    "verbosity": 0,
    "objective": "survival:aft",
    "eval_metric": "aft-nloglik",
    "tree_method": "hist",
}  # Hyperparameters common to all trials

feature_combos = [
    list(
        permanent_teeth.columns.difference(
            ["LOWER", "UPPER", "DOG", "CENS"], sort=False
        )
    ),  # alle features
    [
        "DEVELOPM_STAGE",
        "MAX_MAND",
        "SIDE",
        "POSITION",
        "SEX",
        "BREED_SIZE",
        "SKULL_TYPE_Brachy",
        "SKULL_TYPE_Doligo",
        "SKULL_TYPE_Meso",
        "SKULL_TYPE_Meso*",
        "SKULL_TYPE_Meso**",
        "BREED_CLADE_A",
        "BREED_CLADE_B",
        "BREED_CLADE_D",
        "BREED_CLADE_E",
        "BREED_CLADE_F",
        "BREED_CLADE_H",
        "BREED_CLADE_I",
        "BREED_CLADE_L",
        "BREED_CLADE_M",
        "BREED_CLADE_N",
        "BREED_CLADE_O",
        "BREED_CLADE_P",
        "BREED_CLADE_Q",
        "BREED_CLADE_R",
        "BREED_CLADE_S",
        "BREED_CLADE_T",
        "BREED_CLADE_U",
        "BREED_CLADE_V",
        "BREED_CLADE_W",
        "BREED_CLADE_X",
        "BREED_CLADE_Y",
    ],
    [
        "DEVELOPM_STAGE",
        "TOOTH_NUMBER",
        "SEX",
        "BREED_SIZE",
        "SKULL_TYPE_Brachy",
        "SKULL_TYPE_Doligo",
        "SKULL_TYPE_Meso",
        "SKULL_TYPE_Meso*",
        "SKULL_TYPE_Meso**",
        "BREED_CLADE_A",
        "BREED_CLADE_B",
        "BREED_CLADE_D",
        "BREED_CLADE_E",
        "BREED_CLADE_F",
        "BREED_CLADE_H",
        "BREED_CLADE_I",
        "BREED_CLADE_L",
        "BREED_CLADE_M",
        "BREED_CLADE_N",
        "BREED_CLADE_O",
        "BREED_CLADE_P",
        "BREED_CLADE_Q",
        "BREED_CLADE_R",
        "BREED_CLADE_S",
        "BREED_CLADE_T",
        "BREED_CLADE_U",
        "BREED_CLADE_V",
        "BREED_CLADE_W",
        "BREED_CLADE_X",
        "BREED_CLADE_Y",
    ],
    [
        "TOOTH_NUMBER",
        "MAX_MAND",
        "SIDE",
        "POSITION",
        "DEVELOPM_STAGE",
        "BREED_SIZE",
        "SKULL_TYPE_Brachy",
        "SKULL_TYPE_Doligo",
        "SKULL_TYPE_Meso",
        "SKULL_TYPE_Meso*",
        "SKULL_TYPE_Meso**",
    ],
    [
        "DEVELOPM_STAGE",
        "TOOTH_NUMBER",
        "BREED_SIZE",
        "SKULL_TYPE_Brachy",
        "SKULL_TYPE_Doligo",
        "SKULL_TYPE_Meso",
        "SKULL_TYPE_Meso*",
        "SKULL_TYPE_Meso**",
    ],
]

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3
test_preds = {}

lodo = StratifiedGroupKFold(N_OUTER_FOLDS, shuffle=True, random_state=42)
for i, (train_index, test_index) in enumerate(
    lodo.split(
        X=permanent_teeth, y=permanent_teeth["CENS"], groups=permanent_teeth["DOG"]
    )
):
    print(i)
    train_outer = permanent_teeth.iloc[train_index]
    test_outer = permanent_teeth.iloc[test_index]

    def objective(trial):
        feature_selection = trial.suggest_int(
            "feature_selection", 0, len(feature_combos) - 1
        )

        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
            "aft_loss_distribution": trial.suggest_categorical(
                "aft_loss_distribution", ["normal", "logistic", "extreme"]
            ),
            "aft_loss_distribution_scale": trial.suggest_float(
                "aft_loss_distribution_scale", 0.01, 5, log=True
            ),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "min_child_weight": trial.suggest_float(
                "min_child_weight", 1, 10, log=True
            ),
            "lambda": trial.suggest_float("lambda", 1e-3, 10.0),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        }  # Search space
        params.update(base_params)
        pruning_callback = optuna.integration.XGBoostPruningCallback(
            trial, "valid-aft-nloglik"
        )

        # CV
        train_events = permanent_teeth.loc[train_index, "CENS"]
        train_ids = permanent_teeth.loc[train_index, "DOG"]
        best_iters = []
        sgkf = StratifiedGroupKFold(
            n_splits=N_INNER_FOLDS, shuffle=True, random_state=42
        )
        CV_scores = []

        for idx, (train_idx_inner, test_idx_inner) in enumerate(
            sgkf.split(train_outer, train_events, train_ids)
        ):
            train_inner, test_inner = (
                train_outer.iloc[train_idx_inner],
                train_outer.iloc[test_idx_inner],
            )

            dtrain_inner = make_dmatrix(train_inner, feature_combos[feature_selection])
            dtest_inner = make_dmatrix(test_inner, feature_combos[feature_selection])

            if idx == 0:
                bst_inner = xgb.train(
                    params,
                    dtrain_inner,
                    num_boost_round=1000,
                    evals=[(dtrain_inner, "train"), (dtest_inner, "valid")],
                    early_stopping_rounds=100,
                    verbose_eval=False,
                    callbacks=[pruning_callback],
                )
            else:
                bst_inner = xgb.train(
                    params,
                    dtrain_inner,
                    num_boost_round=1000,
                    evals=[(dtrain_inner, "train"), (dtest_inner, "valid")],
                    early_stopping_rounds=100,
                    verbose_eval=False,
                )

            best_iters.append(bst_inner.best_iteration)
            CV_scores.append(bst_inner.best_score)

        trial.set_user_attr("best_iteration", int(np.median(best_iters)))
        return np.mean(CV_scores)

    # Optuna for hyperparam tuning
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=100, show_progress_bar=False)

    # get best params out of inner loop
    best_iter = study.best_trial.user_attrs["best_iteration"]
    best_params = {}
    best_params.update(base_params)
    best_params.update(study.best_trial.params)
    best_features = best_params.pop("feature_selection")
    best_dist = best_params["aft_loss_distribution"]
    best_scale = best_params["aft_loss_distribution_scale"]

    # dmatrix maken
    dtrain_outer = make_dmatrix(train_outer, feature_combos[best_features])
    dtest_outer = make_dmatrix(test_outer, feature_combos[best_features])

    # outer model trainen
    bst_outer = xgb.train(
        best_params,
        dtrain_outer,
        num_boost_round=best_iter,
        verbose_eval=False,
    )
    preds_outer = bst_outer.predict(dtest_outer, output_margin=True)
    # Store by original indices
    for idx, pred_outer in zip(test_index, preds_outer):
        test_preds[idx] = {
            "pred": pred_outer,
            "dist": best_dist,
            "scale": best_scale,
        }

[I 2026-03-11 09:20:08,800] A new study created in memory with name: no-name-05d65bb0-3922-417b-b917-ab31b9ab5a97


0


[I 2026-03-11 09:20:10,388] Trial 0 finished with value: 1.6343856387990832 and parameters: {'feature_selection': 2, 'learning_rate': 0.29646108753185685, 'aft_loss_distribution': 'normal', 'aft_loss_distribution_scale': 0.11375434320781819, 'max_depth': 6, 'min_child_weight': 6.303712741023154, 'lambda': 1.0304714908772892, 'subsample': 0.5249529423249324, 'colsample_bylevel': 0.977020101086538}. Best is trial 0 with value: 1.6343856387990832.
[I 2026-03-11 09:20:12,570] Trial 1 finished with value: 1.7831868907605053 and parameters: {'feature_selection': 4, 'learning_rate': 0.08974953631146922, 'aft_loss_distribution': 'extreme', 'aft_loss_distribution_scale': 0.05542613116734839, 'max_depth': 6, 'min_child_weight': 9.76964913060929, 'lambda': 5.011211609464354, 'subsample': 0.7355922109715305, 'colsample_bylevel': 0.5142861988214704}. Best is trial 0 with value: 1.6343856387990832.
[I 2026-03-11 09:20:13,662] Trial 2 finished with value: 16.07712170630062 and parameters: {'feature_s

1


[I 2026-03-11 09:21:02,622] Trial 0 finished with value: 3.371140829890161 and parameters: {'feature_selection': 0, 'learning_rate': 0.0011302928951721235, 'aft_loss_distribution': 'normal', 'aft_loss_distribution_scale': 1.1870175374260572, 'max_depth': 4, 'min_child_weight': 1.2331639705795985, 'lambda': 7.190080782196949, 'subsample': 0.9794681880689419, 'colsample_bylevel': 0.5137005894492187}. Best is trial 0 with value: 3.371140829890161.
[I 2026-03-11 09:21:17,248] Trial 1 finished with value: 2.225819147156124 and parameters: {'feature_selection': 0, 'learning_rate': 0.004305584822404167, 'aft_loss_distribution': 'normal', 'aft_loss_distribution_scale': 0.5325161094638812, 'max_depth': 7, 'min_child_weight': 1.425219152418018, 'lambda': 2.6888899493539653, 'subsample': 0.5168212596201548, 'colsample_bylevel': 0.5221203386931623}. Best is trial 1 with value: 2.225819147156124.
[I 2026-03-11 09:21:32,061] Trial 2 finished with value: 2.769647097322951 and parameters: {'feature_se

...

[I 2026-03-11 09:22:17,156] Trial 0 finished with value: 17.178379118494334 and parameters: {'feature_selection': 3, 'learning_rate': 0.010331635746723699, 'aft_loss_distribution': 'normal', 'aft_loss_distribution_scale': 0.02063142452202806, 'max_depth': 7, 'min_child_weight': 1.0320075428473225, 'lambda': 8.75621552595264, 'subsample': 0.6602948261973423, 'colsample_bylevel': 0.5347139889257094}. Best is trial 0 with value: 17.178379118494334.
[I 2026-03-11 09:22:18,366] Trial 1 finished with value: 17.178379118494334 and parameters: {'feature_selection': 3, 'learning_rate': 0.002205887869601389, 'aft_loss_distribution': 'normal', 'aft_loss_distribution_scale': 0.013768994175133461, 'max_depth': 7, 'min_child_weight': 8.293107334151259, 'lambda': 1.9327495412069937, 'subsample': 0.8371626337152116, 'colsample_bylevel': 0.9824824255431018}. Best is trial 0 with value: 17.178379118494334.
[I 2026-03-11 09:22:19,407] Trial 2 finished with value: 17.178379118494334 and parameters: {'feat

3


[I 2026-03-11 09:24:04,699] Trial 0 finished with value: 17.78061512461036 and parameters: {'feature_selection': 0, 'learning_rate': 0.03754601097345545, 'aft_loss_distribution': 'extreme', 'aft_loss_distribution_scale': 0.062024962857402444, 'max_depth': 5, 'min_child_weight': 3.3912096273385544, 'lambda': 9.925290671123763, 'subsample': 0.7574400999051707, 'colsample_bylevel': 0.5156724874039216}. Best is trial 0 with value: 17.78061512461036.
[I 2026-03-11 09:24:14,265] Trial 1 finished with value: 3.1556436736298044 and parameters: {'feature_selection': 4, 'learning_rate': 0.00763811093583154, 'aft_loss_distribution': 'logistic', 'aft_loss_distribution_scale': 2.4955726898738315, 'max_depth': 5, 'min_child_weight': 6.0250094530403855, 'lambda': 1.5078402667396364, 'subsample': 0.9607971562530377, 'colsample_bylevel': 0.5759962178785225}. Best is trial 1 with value: 3.1556436736298044.
[I 2026-03-11 09:24:18,116] Trial 2 finished with value: 2.6625120991485773 and parameters: {'feat

4


[I 2026-03-11 09:25:43,618] Trial 0 finished with value: 17.17433466763926 and parameters: {'feature_selection': 4, 'learning_rate': 0.03034530519001139, 'aft_loss_distribution': 'extreme', 'aft_loss_distribution_scale': 0.019852227268646886, 'max_depth': 4, 'min_child_weight': 2.597635024860083, 'lambda': 3.2442413938237062, 'subsample': 0.5977561411751722, 'colsample_bylevel': 0.9060206065648724}. Best is trial 0 with value: 17.17433466763926.
[I 2026-03-11 09:25:45,634] Trial 1 finished with value: 17.17433466763926 and parameters: {'feature_selection': 1, 'learning_rate': 0.004638250638045571, 'aft_loss_distribution': 'extreme', 'aft_loss_distribution_scale': 0.0700296394727313, 'max_depth': 5, 'min_child_weight': 3.6979941614596337, 'lambda': 2.4999124284575363, 'subsample': 0.5481210639552625, 'colsample_bylevel': 0.9046745357652638}. Best is trial 0 with value: 17.17433466763926.
[I 2026-03-11 09:25:48,302] Trial 2 finished with value: 2.7086677765638587 and parameters: {'featur

In [20]:
# Convert to DataFrame in original order
test_preds_df = pd.DataFrame.from_dict(test_preds, orient="index").sort_index()

In [ ]:
test_preds_df.to_csv("xgb_tooth_preds_perma.csv", sep=";")